In [1]:
import joblib
import pandas as pd
import numpy as np
import os
 

In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
MODEL_PATH = "models/linear_regression.pkl"   # change to test different models
DATASET_PATH = "data/processed/modelling_dataset.csv"
 
# Feature lists (must match training script)
LINEAR_FEATURES = [
    "year", "month_num", "quarter", "month_sin", "month_cos", "season",
    "crime_lag_1", "crime_lag_3", "crime_lag_12",
    "crime_rolling_3", "crime_rolling_6",
    "imd_score",
    "population_2023", "population_density_per_km2",
    "median_annual_earnings_2023", "overcrowding_rate",
]
 
TREE_FEATURES = [
    "year", "month_num", "quarter", "month_sin", "month_cos", "season",
    "crime_lag_1", "crime_lag_3", "crime_lag_12",
    "crime_rolling_3", "crime_rolling_6",
    "imd_score", "income_deprivation_score", "employment_deprivation_score",
    "crime_deprivation_score",
    "population_2023", "population_density_per_km2",
    "claimant_count_rate_2023", "median_annual_earnings_2023",
    "median_house_price_2023", "overcrowding_rate",
]
 

In [3]:
# Auto-detect which feature set to use based on model filename
USE_LINEAR_FEATURES = "linear" in MODEL_PATH.lower()
FEATURES = LINEAR_FEATURES if USE_LINEAR_FEATURES else TREE_FEATURES
 
# ─────────────────────────────────────────────────────────────────────────────
# LOAD MODEL AND DATA
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 60)
print("  CRIME PREDICTION MODEL — TEST SUITE")
print("=" * 60)
print(f"\nLoading model: {MODEL_PATH}")
model = joblib.load(MODEL_PATH)
 
print(f"Loading dataset: {DATASET_PATH}")
df = pd.read_csv(DATASET_PATH)
df["date"] = pd.to_datetime(df["date"])
df = df.dropna(subset=["crime_lag_1", "crime_lag_3", "crime_lag_12",
                        "crime_rolling_3", "crime_rolling_6"]).reset_index(drop=True)
print(f"  Rows: {len(df)} | Boroughs: {df['borough'].nunique()} | "
      f"Period: {df['date'].min().strftime('%Y-%m')} to {df['date'].max().strftime('%Y-%m')}")
 

  CRIME PREDICTION MODEL — TEST SUITE

Loading model: models/linear_regression.pkl
Loading dataset: data/processed/modelling_dataset.csv
  Rows: 792 | Boroughs: 33 | Period: 2024-03 to 2026-02


In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# HELPER: Build a feature row for any borough/date
# ─────────────────────────────────────────────────────────────────────────────
SEASON_MAP = {
    12: "winter", 1: "winter", 2: "winter",
    3: "spring", 4: "spring", 5: "spring",
    6: "summer", 7: "summer", 8: "summer",
    9: "autumn", 10: "autumn", 11: "autumn",
}
 
def build_feature_row(history: pd.DataFrame, target_date: pd.Timestamp, borough: str) -> pd.DataFrame:
    """
    Build a single-row DataFrame for prediction at target_date for a borough.
    Uses the borough's history to compute lag and rolling features.
    """
    h = history[history["borough"] == borough].sort_values("date").reset_index(drop=True)
    if len(h) == 0:
        raise ValueError(f"No history found for borough: {borough}")
 
    # Socioeconomic features (constant per borough — take from any row)
    socio_cols = [
        "imd_score", "income_deprivation_score", "employment_deprivation_score",
        "crime_deprivation_score",
        "population_2023", "population_density_per_km2",
        "claimant_count_rate_2023", "median_annual_earnings_2023",
        "median_house_price_2023", "overcrowding_rate",
    ]
    socio = h.iloc[-1][socio_cols].to_dict()
 
    # Lag features — use most recent values
    lag_1  = h.iloc[-1]["crime_count"]
    lag_3  = h.iloc[-3]["crime_count"]  if len(h) >= 3  else lag_1
    lag_12 = h.iloc[-12]["crime_count"] if len(h) >= 12 else lag_1
    rolling_3 = h.iloc[-3:]["crime_count"].mean() if len(h) >= 3 else lag_1
    rolling_6 = h.iloc[-6:]["crime_count"].mean() if len(h) >= 6 else lag_1
 
    # Temporal features
    month_num = target_date.month
    row = {
        "year":          target_date.year,
        "month_num":     month_num,
        "quarter":       (month_num - 1) // 3 + 1,
        "month_sin":     np.sin(2 * np.pi * month_num / 12),
        "month_cos":     np.cos(2 * np.pi * month_num / 12),
        "season":        SEASON_MAP[month_num],
        "crime_lag_1":   lag_1,
        "crime_lag_3":   lag_3,
        "crime_lag_12":  lag_12,
        "crime_rolling_3": rolling_3,
        "crime_rolling_6": rolling_6,
        **socio,
    }
    return pd.DataFrame([row])
 
 
def forecast_recursive(model, history: pd.DataFrame, borough: str, num_months: int) -> list:
    """
    Recursively forecast `num_months` ahead.
    Each prediction becomes the basis for the next month's lag features.
    """
    h = history.copy()
    forecasts = []
    last_date = h[h["borough"] == borough]["date"].max()
 
    for _ in range(num_months):
        target_date = last_date + pd.DateOffset(months=1)
        X = build_feature_row(h, target_date, borough)
        pred = float(model.predict(X[FEATURES])[0])
        forecasts.append({"date": target_date, "prediction": pred})
 
        # Add this prediction as a new "historical" row for the next iteration
        new_row = X.iloc[0].to_dict()
        new_row["borough"]     = borough
        new_row["date"]        = target_date
        new_row["month"]       = target_date.strftime("%Y-%m")
        new_row["crime_count"] = pred
        h = pd.concat([h, pd.DataFrame([new_row])], ignore_index=True)
        last_date = target_date
 
    return forecasts
 
 

In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# DEMO 1: Quick predictions on known test data
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  DEMO 1: Quick predictions on test data")
print("  (model predicts → we compare to known actual)")
print("=" * 60)
 
# Pick 5 random rows from the last 5 months (test period)
test_start = sorted(df["date"].unique())[int(len(df["date"].unique()) * 0.8)]
test_data = df[df["date"] >= test_start]
sample = test_data.sample(n=5, random_state=42).sort_values(["borough", "date"])
 
print(f"\n{'Borough':<25} {'Month':<10} {'Actual':>10} {'Predicted':>10} {'Error':>10} {'%':>6}")
print("-" * 75)
total_error_pct = 0
for _, row in sample.iterrows():
    X = row[FEATURES].to_frame().T
    pred = float(model.predict(X)[0])
    actual = row["crime_count"]
    error = abs(actual - pred)
    pct = error / actual * 100
    total_error_pct += pct
    print(f"{row['borough']:<25} {row['month']:<10} {actual:>10,.0f} {pred:>10,.0f} "
          f"{error:>10,.0f} {pct:>5.1f}%")
print("-" * 75)
print(f"{'Average error':<25} {' ':<10} {' ':>10} {' ':>10} {' ':>10} {total_error_pct/5:>5.1f}%")
 


  DEMO 1: Quick predictions on test data
  (model predicts → we compare to known actual)

Borough                   Month          Actual  Predicted      Error      %
---------------------------------------------------------------------------
Hackney                   2025-10         3,425      3,609        184   5.4%
Kensington and Chelsea    2025-10         2,351      2,481        130   5.5%
Merton                    2025-10         1,409      1,491         82   5.8%
Richmond upon Thames      2025-11         1,123      1,299        176  15.7%
Southwark                 2025-10         4,220      4,417        197   4.7%
---------------------------------------------------------------------------
Average error                                                           7.4%


In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# DEMO 2: Forecasting future months
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  DEMO 2: Forecasting next 6 months")
print("  (predicting months beyond the dataset)")
print("=" * 60)
 
last_date = df["date"].max()
print(f"\nLast actual data point in dataset: {last_date.strftime('%Y-%m')}")
print(f"Forecasting: {(last_date + pd.DateOffset(months=1)).strftime('%Y-%m')} "
      f"to {(last_date + pd.DateOffset(months=6)).strftime('%Y-%m')}\n")
 
# Forecast for 3 representative boroughs
demo_boroughs = ["Westminster", "Camden", "Bexley"]
for borough in demo_boroughs:
    forecasts = forecast_recursive(model, df, borough, num_months=6)
    last_actual = df[df["borough"] == borough].sort_values("date").iloc[-1]["crime_count"]
 
    print(f"\n  {borough}")
    print(f"    Last actual ({last_date.strftime('%Y-%m')}): {last_actual:,.0f} crimes")
    print(f"    Forecast:")
    for f in forecasts:
        print(f"      {f['date'].strftime('%Y-%m')}  →  {f['prediction']:>7,.0f} crimes")
 
print("\n  Note: Recursive forecasting compounds error over time.")
print("        Forecasts more than 3-6 months out should be interpreted with caution.")
 


  DEMO 2: Forecasting next 6 months
  (predicting months beyond the dataset)

Last actual data point in dataset: 2026-02
Forecasting: 2026-03 to 2026-08


  Westminster
    Last actual (2026-02): 6,450 crimes
    Forecast:
      2026-03  →    7,393 crimes
      2026-04  →    7,509 crimes
      2026-05  →    7,545 crimes
      2026-06  →    7,643 crimes
      2026-07  →    7,809 crimes
      2026-08  →    7,668 crimes

  Camden
    Last actual (2026-02): 3,743 crimes
    Forecast:
      2026-03  →    3,920 crimes
      2026-04  →    4,028 crimes
      2026-05  →    4,148 crimes
      2026-06  →    4,143 crimes
      2026-07  →    4,342 crimes
      2026-08  →    4,179 crimes

  Bexley
    Last actual (2026-02): 1,472 crimes
    Forecast:
      2026-03  →    1,603 crimes
      2026-04  →    1,711 crimes
      2026-05  →    1,766 crimes
      2026-06  →    1,734 crimes
      2026-07  →    1,835 crimes
      2026-08  →    1,785 crimes

  Note: Recursive forecasting compounds error over ti

In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# DEMO 3: Interactive prediction tool
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  DEMO 3: Interactive prediction tool")
print("=" * 60)
 
boroughs = sorted(df["borough"].unique())
 
print("\nAvailable boroughs:")
for i in range(0, len(boroughs), 3):
    line = ""
    for j in range(3):
        if i + j < len(boroughs):
            line += f"  {i+j+1:>2}. {boroughs[i+j]:<25}"
    print(line)
 
print("\n" + "─" * 60)
print("  Type a borough number and a target month to get a prediction.")
print("  Type 'q' to quit.")
print("─" * 60)
 
while True:
    print()
    choice = input("Borough number (1-33) or 'q': ").strip().lower()
    if choice == "q":
        print("\nGoodbye! 👋")
        break
    try:
        borough = boroughs[int(choice) - 1]
    except (ValueError, IndexError):
        print("  Invalid choice. Please enter a number 1-33 or 'q'.")
        continue
 
    month_str = input(f"Target month for {borough} (YYYY-MM, e.g. 2026-06): ").strip()
    try:
        target_dt = pd.to_datetime(month_str + "-01")
    except (ValueError, TypeError):
        print("  Invalid format. Use YYYY-MM (e.g. 2026-06)")
        continue
 
    # Determine if historical or future
    last_date_borough = df[df["borough"] == borough]["date"].max()
    earliest_date_borough = df[df["borough"] == borough]["date"].min()
 
    if target_dt < earliest_date_borough:
        print(f"  Date too early. Earliest available: {earliest_date_borough.strftime('%Y-%m')}")
        continue
 
    if target_dt <= last_date_borough:
        # Historical — we have the actual value
        actual_row = df[(df["borough"] == borough) & (df["date"] == target_dt)]
        if len(actual_row) == 0:
            print("  No data available for that month (likely in early period before lag features).")
            continue
        X = actual_row[FEATURES]
        pred = float(model.predict(X)[0])
        actual = float(actual_row["crime_count"].iloc[0])
        error = abs(actual - pred)
        pct = error / actual * 100
        print(f"\n  ✓ Historical prediction:")
        print(f"    {borough} — {month_str}")
        print(f"    Predicted : {pred:>8,.0f} crimes")
        print(f"    Actual    : {actual:>8,.0f} crimes")
        print(f"    Error     : {error:>8,.0f} ({pct:.1f}%)")
    else:
        # Future — recursive forecast
        months_ahead = (target_dt.year - last_date_borough.year) * 12 + \
                       (target_dt.month - last_date_borough.month)
        if months_ahead > 12:
            print(f"  Forecasts beyond 12 months are unreliable. "
                  f"You requested {months_ahead} months ahead.")
            continue
        forecasts = forecast_recursive(model, df, borough, months_ahead)
        pred = forecasts[-1]["prediction"]
        print(f"\n  ⚡ Future forecast ({months_ahead} months ahead):")
        print(f"    {borough} — {month_str}")
        print(f"    Predicted : {pred:>8,.0f} crimes")
        if months_ahead > 3:
            print(f"    ⚠️  Note: forecasts more than 3 months ahead lose accuracy")



  DEMO 3: Interactive prediction tool

Available boroughs:
   1. Barking and Dagenham        2. Barnet                      3. Bexley                   
   4. Brent                       5. Bromley                     6. Camden                   
   7. City of London              8. Croydon                     9. Ealing                   
  10. Enfield                    11. Greenwich                  12. Hackney                  
  13. Hammersmith and Fulham     14. Haringey                   15. Harrow                   
  16. Havering                   17. Hillingdon                 18. Hounslow                 
  19. Islington                  20. Kensington and Chelsea     21. Kingston upon Thames     
  22. Lambeth                    23. Lewisham                   24. Merton                   
  25. Newham                     26. Redbridge                  27. Richmond upon Thames     
  28. Southwark                  29. Sutton                     30. Tower Hamlets            


Borough number (1-33) or 'q':  2
Target month for Barnet (YYYY-MM, e.g. 2026-06):  2028-06


  Forecasts beyond 12 months are unreliable. You requested 28 months ahead.



Borough number (1-33) or 'q':  2
Target month for Barnet (YYYY-MM, e.g. 2026-06):  2027-03


  Forecasts beyond 12 months are unreliable. You requested 13 months ahead.



Borough number (1-33) or 'q':  2
Target month for Barnet (YYYY-MM, e.g. 2026-06):  2027-01



  ⚡ Future forecast (11 months ahead):
    Barnet — 2027-01
    Predicted :    2,931 crimes
    ⚠️  Note: forecasts more than 3 months ahead lose accuracy



Borough number (1-33) or 'q':  q



Goodbye! 👋
